In [64]:
from datasets import load_dataset
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
import random
import pandas as pd
import datetime

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ninahew/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [65]:
import numpy as np
desired_width=320
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns',10)

In [66]:
def generate_dict(x, lang):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    print('dataset ' + '{lang}'.format(lang=lang) + ' loaded')
    # Consider returning number of rows in datasets
    corpus_lang = lang_ds['train']['text'][0:x]
    # print(corpus_lang)
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    tokenized_lang= word_tokenize(string_lang)
    print("{lang}".format(lang=lang) + " after tokenisation")
    print("{lang}".format(lang=lang), " : ", str(len(tokenized_lang)), " words")
    # Remove punctuation, proper nouns and numerical entries
    cleaned_lang = [word for word in tokenized_lang if ((len(word) > 1) and (not(word[0].isupper())) and (not(word.isnumeric())))]
    print("{lang}".format(lang=lang) + " after data cleaning")
    print("{lang}".format(lang=lang), " : ", str(len(cleaned_lang)), " words")
    dict_lang = {}
    for word in cleaned_lang:
        if word not in dict_lang:
            dict_lang[word] = 0
        dict_lang[word] += 1
    return dict_lang


In [67]:
vls = generate_dict(100, 'vls')
nl = generate_dict(100, 'nl')


dataset vls loaded
vls after tokenisation
vls  :  37159  words
vls after data cleaning
vls  :  22845  words
dataset nl loaded
nl after tokenisation
nl  :  213125  words
nl after data cleaning
nl  :  149607  words


In [68]:
# print(type(nl))

In [69]:
corpus_vls_nl = {'west flemish' : vls, 'nederlands' : nl}


In [70]:
# print(corpus_af_nl)

In [71]:
# Creating test function using small sample data, before using wikipedia data

# corpus_1 = {'en' : {'cat' : 2, 'dog' : 5, 'lizard' : 7},
#           'fr' : {'chat' : 4, 'chien' : 2, 'lizard' : 10}}

# Function takes a word and a corpus and returns its language
def identify_language(given_word, corpus):
    current_language = 'unknown'
    highest_frequency = 0
    for language_name, word_frequencies in corpus.items():
        if given_word in word_frequencies:
            if word_frequencies[given_word] > highest_frequency:
                current_language = language_name
                highest_frequency = word_frequencies[given_word]

    return current_language, highest_frequency




In [72]:
def generate_test_phrase(x: int, lang: str, phrase_length: int):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    # print('dataset loaded')
    corpus_lang = lang_ds['train']['text'][0:x]
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    # print(str(len(string_lang)) + ' words in string_lang')
    tokenized_lang= word_tokenize(string_lang)
    # print(str(len(tokenized_lang)) + ' words in token lang')
    cleaned_lang = [word for word in tokenized_lang if (len(word) > 1)]
    # print(str(len(cleaned_lang)) + 'words in clean lang')
    no_proper_nouns_lang = [word for word in cleaned_lang if not word[0].isupper()]
    no_numbers_lang = [word for word in no_proper_nouns_lang if not word.isnumeric()]
    random_number = random.randint(0,(len(no_numbers_lang) - phrase_length))
    return no_numbers_lang[random_number : random_number+phrase_length]

In [73]:
w_flemish_test_phrase_1 = generate_test_phrase(100, 'vls', 10)
print(w_flemish_test_phrase_1)

['trekkn', 'veel', 'dagsjestoeristn', 'veel', 'protestantn', 'trouwdegen', 'met', 'andere', 'protestantn', 'van']


In [74]:
[identify_language(word, corpus_vls_nl) for word in w_flemish_test_phrase_1]

[('west flemish', 5),
 ('nederlands', 236),
 ('west flemish', 1),
 ('nederlands', 236),
 ('west flemish', 4),
 ('west flemish', 1),
 ('nederlands', 1492),
 ('nederlands', 368),
 ('west flemish', 4),
 ('nederlands', 7101)]

In [75]:
nederlands_test_phrase_1 = generate_test_phrase(100, 'nl', 10)
print(nederlands_test_phrase_1)

['en', 'jonkheer', 'een', 'weddenschap', 'af', 'dat', 'hij', 'de', 'zo', 'ver']


In [76]:
[identify_language(word, corpus_vls_nl) for word in nederlands_test_phrase_1]

[('nederlands', 5248),
 ('nederlands', 1),
 ('nederlands', 4088),
 ('nederlands', 1),
 ('nederlands', 121),
 ('nederlands', 1055),
 ('nederlands', 696),
 ('nederlands', 12367),
 ('nederlands', 130),
 ('nederlands', 19)]

In [77]:
result = [identify_language(word, corpus_vls_nl) for word in nederlands_test_phrase_1]
# print(result)


# print(result[0][0])
language_occurrence = [lang for lang, freq in result]
s = pd.Series(language_occurrence)
# print(s.mode().to_string().strip())


In [78]:
def sentence_language_identifier(sentence: list[str], corpus):
    list_languages = [identify_language(word, corpus) for word in sentence]
    language_occurrence = [lang for lang, freq in list_languages]
    # print(language_occurrence)
    lang_occ_series = pd.Series(language_occurrence)
    # print(lang_occ_series)
    modal_lang = lang_occ_series.mode()
    if modal_lang.empty:
        return 'unknown'
    elif 'nederland' in modal_lang.to_string():
        return 'nl'
    else:
        return 'vls'

In [79]:
nl1 = generate_test_phrase(10, 'nl', 10)
sentence_language_identifier(nl1, corpus_vls_nl)

'nl'

In [80]:
vls = generate_test_phrase(10, 'vls', 10)
sentence_language_identifier(vls, corpus_vls_nl)

'vls'

In [81]:
# Testing dutch sentences

results = {}

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'nl', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time = datetime.datetime.now()
        results['Corpus size'] = corpus_size
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Corpus size': 1, 'Sentence_length': 1, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=512)}
{'Corpus size': 1, 'Sentence_length': 2, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=569)}
{'Corpus size': 1, 'Sentence_length': 3, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=517)}
{'Corpus size': 1, 'Sentence_length': 4, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=533)}
{'Corpus size': 1, 'Sentence_length': 5, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=523)}
{'Corpus size': 1, 'Sentence_length': 6, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=524)}
{'Corpus size': 1, 'Sentence_length': 7, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=509)}
{'Corpus size': 1, 'Sentence_length': 8, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=528)}
{'Corpus size': 1, 'Sentence_length': 9, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=513)}
{

In [82]:
# Testing Flemish sentences

results = {}

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'vls', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time = datetime.datetime.now()
        results['Corpus size'] = corpus_size
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Corpus size': 1, 'Sentence_length': 1, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=606)}
{'Corpus size': 1, 'Sentence_length': 2, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=757)}
{'Corpus size': 1, 'Sentence_length': 3, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=813)}
{'Corpus size': 1, 'Sentence_length': 4, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=605)}
{'Corpus size': 1, 'Sentence_length': 5, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=613)}
{'Corpus size': 1, 'Sentence_length': 6, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=588)}
{'Corpus size': 1, 'Sentence_length': 7, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=604)}
{'Corpus size': 1, 'Sentence_length': 8, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=600)}
{'Corpus size': 1, 'Sentence_length': 9, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=5

In [83]:
from nltk.classify.textcat import TextCat

In [84]:
# # refactoring existing LID tool to include TextCat results in the output, testing both models on the same data
#
# # Benchmarking Flemish sentences
# tc = TextCat()
#
#
# results = {}
#
# for corpus_size in range(1,6):
#     for i in range(1,11):
#         test_sentence = generate_test_phrase(corpus_size ,'vls', i)
#         start_time_rule_based = datetime.datetime.now()
#         predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
#         end_time_rule_based = datetime.datetime.now()
#         results['Corpus size'] = corpus_size
#         results['Sentence_length'] = len(test_sentence)
#         results['Language (RB)'] = predicted_language_rule_based
#         results['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
#         test_sentence_joined = " ".join(test_sentence)
#         start_time_textcat = datetime.datetime.now()
#         predicted_language_textcat = tc.guess_language(test_sentence_joined)
#         end_time_textcat = datetime.datetime.now()
#         results['Language (TC)'] = predicted_language_textcat
#         results['Time taken (TC)'] = end_time_textcat - start_time_rule_based
#         print(results)
#


In [85]:
# Benchmarking Dutch sentences
tc = TextCat()

results_list = []

for corpus_size in range(1,6):
    for i in range(1,11):
        result_dict = {}
        test_sentence = generate_test_phrase(corpus_size ,'nl', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Corpus size'] = corpus_size
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)

df = pd.DataFrame(results_list)
# print(df)



In [86]:
print(df.groupby(['Language (RB)']).count())

               Corpus size  Sentence_length  Time taken (RB)  Language (TC)  Time taken (TC)
Language (RB)                                                                               
nl                      50               50               50             50               50


In [87]:
print(df.groupby(['Language (TC)']).count())

               Corpus size  Sentence_length  Language (RB)  Time taken (RB)  Time taken (TC)
Language (TC)                                                                               
afr                     14               14             14               14               14
eng                      2                2              2                2                2
eng                      1                1              1                1                1
fri                      1                1              1                1                1
fur                      1                1              1                1                1
glv                      1                1              1                1                1
hrv                      1                1              1                1                1
kos                      2                2              2                2                2
lad                      1                1              1            

In [88]:
# Benchmarking Flemish sentences
tc = TextCat()

results_list = []

for corpus_size in range(1,6):
    for i in range(1,11):
        result_dict = {}
        test_sentence = generate_test_phrase(corpus_size ,'vls', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Corpus size'] = corpus_size
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)


df = pd.DataFrame(results_list)



In [89]:
print(df.groupby(['Language (RB)']).count())

               Corpus size  Sentence_length  Time taken (RB)  Language (TC)  Time taken (TC)
Language (RB)                                                                               
nl                      31               31               31             31               31
vls                     19               19               19             19               19


In [90]:
print(df.groupby(['Language (TC)']).count())

               Corpus size  Sentence_length  Language (RB)  Time taken (RB)  Time taken (TC)
Language (TC)                                                                               
abk                      1                1              1                1                1
afr                      5                5              5                5                5
deu                      1                1              1                1                1
eng                      4                4              4                4                4
fri                      2                2              2                2                2
fur                      1                1              1                1                1
nds                      3                3              3                3                3
nyk                      1                1              1                1                1
sot                      1                1              1            

In [91]:

print(df)

    Corpus size  Sentence_length Language (RB)        Time taken (RB) Language (TC)        Time taken (TC)
0             1                1           vls 0 days 00:00:00.000655           vls 0 days 00:00:00.027670
1             1                2           vls 0 days 00:00:00.000589           sot 0 days 00:00:00.088798
2             1                3            nl 0 days 00:00:00.000574           vls 0 days 00:00:00.095701
3             1                4            nl 0 days 00:00:00.000644           vls 0 days 00:00:00.081483
4             1                5            nl 0 days 00:00:00.000639           vls 0 days 00:00:00.111328
5             1                6            nl 0 days 00:00:00.000612           afr 0 days 00:00:00.173081
6             1                7           vls 0 days 00:00:00.000614           eng 0 days 00:00:00.332045
7             1                8            nl 0 days 00:00:00.000617           vls 0 days 00:00:00.248618
8             1                9     

In [98]:
# Benchmarking Flemish sentences
tc = TextCat()

results_list = []

for corpus_size in range(6,15):
    for i in range(2,11):
        result_dict = {}
        test_sentence = generate_test_phrase(corpus_size ,'vls', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Corpus size'] = corpus_size
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)


df = pd.DataFrame(results_list)



In [99]:
print(df.groupby(['Language (RB)']).count())

               Corpus size  Sentence_length  Time taken (RB)  Language (TC)  Time taken (TC)
Language (RB)                                                                               
nl                      30               30               30             30               30
vls                     51               51               51             51               51


In [100]:
print(df.groupby(['Language (TC)']).count())

               Corpus size  Sentence_length  Language (RB)  Time taken (RB)  Time taken (TC)
Language (TC)                                                                               
afr                      8                8              8                8                8
arg                      1                1              1                1                1
deu                      1                1              1                1                1
eng                      5                5              5                5                5
fri                      1                1              1                1                1
glv                      1                1              1                1                1
lmo                      1                1              1                1                1
nds                      3                3              3                3                3
nob                      1                1              1            